### **라이브러리 로드**

In [ ]:
# 데이터 처리
import pandas as pd
import numpy as np
from IPython.display import display
import pprint

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as mticker

# 통계 검정
import scipy
from scipy.stats import chi2_contingency  # 카이제곱 검정
from scipy.stats import spearmanr         # 스피어만 상관계수
from scipy.stats import PermutationMethod # Monte Carlo 시뮬레이션
from scipy.stats import fisher_exact      # Fisher's exact
from scipy.stats import kruskal           # Kruskal-Wallis
from scikit_posthocs import posthoc_dunn  # Dunn's test

# 경고 메시지 무시
import warnings
warnings.filterwarnings('ignore')

# 운영체제별 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False

# 데이터프레임 출력 제한 해제
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

---
### **데이터 로드**

In [ ]:
# IT 숏츠 영상 분석 결과
df_video = pd.read_csv('../../../data/results/main_dataset/쇼츠 영상(+이미지 분석 정보)/result_sample_shorts_all_for_video_agent_fixed.csv', encoding='utf-8')

# IT 숏츠에서 성공으로 분류된 영상의 댓글 감성 분석 결과
df_comment_success = pd.read_csv('../../../data/results/main_dataset/댓글(필터링+감성분석 정보)/sentiment_filtered_cleaned_it_shorts_success_comment.csv', encoding='utf-8')

# IT 숏츠에서 실패으로 분류된 영상의 댓글 감성 분석 결과
df_comment_fail = pd.read_csv('../../../data/results/main_dataset/댓글(필터링+감성분석 정보)/sentiment_filtered_cleaned_it_shorts_fail_comment.csv', encoding='utf-8')

In [ ]:
# 비디오 분석 데이터_컬럼 확인
print('[숏츠 영상 데이터 분석 결과]')
print(df_video.columns.tolist())
display(df_video.head(1))

In [ ]:
# 댓글 감성분석 데이터_컬럼 및 감성 비율 확인
print('[성공으로 분류된 영상]')
print(df_comment_success.columns.tolist())
print(df_comment_success['sentiment'].value_counts())
print('-'*30)
print('[실패로 분류된 영상]')
print(df_comment_fail.columns.tolist())
print(df_comment_fail['sentiment'].value_counts())

---
### **데이터 전처리**

In [ ]:
# ========================================
# [전처리 1] 필요한 컬럼만 선택
# ========================================

# [역할] 분석에 필요한 컬럼만 추출하여 데이터 크기 축소
# [근거] 불필요한 컬럼을 제거하여 메모리 효율화 및 가독성 향상

VIDEO_COLS = [
    'video_id', 'production_quality', 'lighting_style', 'color_mood',
    'editing_pace', 'motion_graphic', 'video_format', 'first_3sec',
    'background_style', 'person_ratio', 'face_ratio', 'grade', 'domain'
]

COMMENT_COLS = ['comment_id', 'video_id', 'sentiment', 'is_korean', 'is_event']

df_video_slim = df_video[VIDEO_COLS]
df_comment_success_slim = df_comment_success[COMMENT_COLS]
df_comment_fail_slim  = df_comment_fail[COMMENT_COLS]

# ========================================
# [전처리 2] IT 도메인 필터링
# ========================================

# [역할] df_video에서 IT 도메인 영상만 추출
# [근거] df_video에 FnB/IT 두 도메인이 섞여 있으므로
#        IT 분석을 위해 IT만 분리

df_video_it = df_video_slim[df_video_slim['domain'] == 'IT'].reset_index(drop=True)
print(f"IT 영상 수: {len(df_video_it)}개")

# ========================================
# [전처리 3] 이벤트 참여 댓글 제거
# ========================================

# [역할] is_event=True인 댓글 제거
# [근거] 이벤트 참여 목적의 댓글은 영상 콘텐츠에 대한 반응이 아니므로
#        감성 비율을 왜곡할 수 있음

# IT는 실패로 분류된 영상의 댓글이 14개로 분석 불가 → 성공으로 분류된 영상만 사용
before = len(df_comment_success_slim)
df_comment_it = df_comment_success_slim[df_comment_success_slim['is_event'] == False]

print(f"\n이벤트 댓글과 외국어 댓글 제거하기 전 댓글 수: {before}개")
print(f"\n이벤트 댓글 제거: {before - len(df_comment_it)}개 제거 → {len(df_comment_it)}개 남음")

# ========================================
# [전처리 4] 외국어 댓글 제거
# ========================================

# [역할] is_korean=False인 댓글 제거
# [근거] 외국어 댓글은 뉘앙스 파악의 한계로 감성 분류 신뢰도가 낮을 수 있음
#        분석 대상이 한국 브랜드의 유튜브 채널이므로 한국어 댓글만 분석하고자 함

before = len(df_comment_it)
df_comment_it = df_comment_it[df_comment_it['is_korean'] == True].reset_index(drop=True)
print(f"\n외국어 댓글 제거: {before - len(df_comment_it)}개 제거 → {len(df_comment_it)}개 남음")

# ========================================
# [전처리 5] video_id 기준으로 merge
# ========================================

# [역할] 영상 분류 정보에 댓글 감성 분석 결과를 결합
# [근거] 영상 데이터를 기준으로 merge하여 영상 분류 정보를 중심으로 댓글 데이터를 연결
# [작업] how='inner'로 양쪽 데이터에 모두 존재하는 video_id만 결합
#        → 댓글이 없는 영상은 제외됨

df_it = pd.merge(df_video_it, df_comment_it, on='video_id', how='inner')
print(f"\nmerge 후 데이터: {len(df_it)}개")
print(f"[NaN 현황]\n{df_it.isna().sum()}")

# ========================================
# [전처리 6] 최종 데이터 확인
# ========================================

# [역할] 전처리 완료 후 데이터 상태 최종 점검
# [근거] 행/열 수, 감성 분포, 샘플 데이터를 확인하여 전처리 결과 검증

print(f"\n최종 데이터(성공으로 분류된 영상): {len(df_it)}행 / {len(df_it.columns)}열")
print(f"분석대상 비디오 개수: {df_it['video_id'].nunique()}개")
print(f"\n[감성 분포]")
sentiment_counts = df_it['sentiment'].value_counts()
sentiment_pct = df_it['sentiment'].value_counts(normalize=True) * 100
for sentiment in sentiment_counts.index:
    print(f"  {sentiment}: {sentiment_counts[sentiment]}개 ({sentiment_pct[sentiment]:.1f}%)")

display(df_it.head(3))

---
### **통계 검정(단일변수)**

<성공으로 분류된 영상에 대해 분석하기로 선택한 이유>

현재 보유한 IT 숏폼 데이터에서 성공 영상 댓글은 2,931개, 실패 영상 댓글은 14개로 극심한 불균형이 존재한다. 실패 영상 댓글 수가 너무 적어 통계 검정 결과의 신뢰도를 보장하기 어렵기 때문에, 성공 영상 댓글만을 분석 대상으로 설정했다.
이와 같은 방향으로 분석한 뒤, "성공한 영상에서 어떤 콘텐츠 특성이 긍정적인 반응을 이끌어내는가"에 집중하여 동영상 제작 가이드 도출에 활용하면 좋을 것 같다.

### **A. 범주형 변수 → 카이제곱 검정 or Monte Carlo 시뮬레이션**

대상 변수: `video_format`, `production_quality`, `editing_pace`, `color_mood`, `first_3sec`, `motion_graphic`, `background_style`

##### **1단계: Cochran's rule 확인**
아래 두 조건을 모두 충족해야 카이제곱 검정을 적용할 수 있다.
- 기대빈도 1 미만인 셀이 없어야 함
- 기대빈도 5 미만인 셀의 비율이 20% 이하여야 함

두 조건 중 하나라도 위반 시 → Monte Carlo 시뮬레이션으로 대체 예정

##### **2단계: 카이제곱 검정 vs Monte Carlo 시뮬레이션**

| 구분 | 카이제곱 검정 | Monte Carlo 시뮬레이션 |
|---|---|---|
| 적용 조건 | Cochran's rule 충족 | Cochran's rule 위반 |
| 데이터 규모 | 대규모 | 대규모 |
| p-value 방식 | 이론적 분포 기반 | 시뮬레이션 기반 추정 |
| 계산 비용 | 낮음 | 높음 |

<p-value를 구하는 방법의 차이>
카이제곱 검정 → 카이제곱 통계량을 이론적 분포(카이제곱 분포)에 대입해서 p-value 계산
Monte Carlo → 동일한 카이제곱 통계량을 시뮬레이션 결과와 비교해서 p-value 추정

##### **3단계: 크래머 V (효과 크기)**
카이제곱/Monte Carlo 검정 결과가 유의미한 경우에만 적용한다.
카이제곱 통계량을 기반으로 계산하므로 두 검정 방법 모두 동일하게 적용 가능하다.
공식: `√(χ² / (N · min(r−1, c−1)))`
효과 크기 기준은 `df*(= min(r−1, c−1))`에 따라 다르게 적용한다.

| df* | 매우 작음 | 작음 | 중간 | 큼 |
|---|---|---|---|---|
| 1 | ~0.10 | 0.10~0.30 | 0.30~0.50 | 0.50~ |
| 2 | ~0.07 | 0.07~0.21 | 0.21~0.35 | 0.35~ |
| 3 | ~0.06 | 0.06~0.17 | 0.17~0.29 | 0.29~ |
| 4 | ~0.05 | 0.05~0.15 | 0.15~0.25 | 0.25~ |
| 5 | ~0.04 | 0.04~0.13 | 0.13~0.22 | 0.22~ |

##### **4단계: 조정된 잔차 사후검정**
카이제곱/Monte Carlo 검정 결과가 유의미한 경우에만 수행한다.
어떤 셀이 기대보다 유의미하게 많거나 적은지 파악한다.
- 기준: |조정된 잔차| > 1.96 (95% 신뢰수준)
- 양수: 기대보다 많음 (↑)
- 음수: 기대보다 적음 (↓)


### **B. 연속형 변수 → Kruskal-Wallis 검정**

대상 변수: `person_ratio`, `face_ratio`

연속형 변수인 인물/얼굴 등장 비율이 감성 그룹(긍정/중립/부정)에 따라
유의미하게 다른지 확인한다.
카이제곱 검정은 범주형 변수에만 적용 가능하므로,
연속형 변수에는 Kruskal-Wallis 검정을 사용한다.

- 적용 조건: 정규성 가정이 필요 없는 비모수 검정
- 귀무가설: 세 감성 그룹(긍정/중립/부정) 간 ratio 분포에 차이가 없다
- 유의수준: p < 0.05

유의미한 경우 사후검정으로 **Dunn's test**를 수행하여
어떤 그룹 간에 차이가 있는지 파악한다.
- 다중비교 보정: Bonferroni
- 기준: 보정된 p < 0.05

효과크기는 **eta-squared (η²)** 로 측정한다.
- 공식: `η² = (H - k + 1) / (n - k)`
- H: Kruskal-Wallis 검정 통계량 / k: 그룹 수 / n: 전체 샘플 수
- 해석 기준:

| η² | 해석 |
|---|---|
| 0.01 미만 | 매우 작은 효과 |
| 0.01 ~ 0.06 | 작은 효과 |
| 0.06 ~ 0.14 | 중간 효과 |
| 0.14 이상 | 큰 효과 |

---
#### **A. 범주형 변수 → 카이제곱 검정 or Monte Carlo 시뮬레이션**

In [ ]:
# ========================================
# 통계 검정 함수 정의
# ========================================

# [역할] 범주형 변수와 sentiment 간의 관계를 통계적으로 검정
# [작업] Cochran's rule 확인 → 카이제곱 or Monte Carlo 시뮬레이션
#        → 크래머 V → 조정된 잔차 사후검정

def chi2_test(df, col, target='sentiment', alpha=0.05, n_simulations=9999):
    """
    Parameters
    ----------
    df            : 분석 데이터프레임
    col           : 검정할 범주형 변수 컬럼명
    target        : 감성 컬럼명 (기본값: 'sentiment')
    alpha         : 유의수준 (기본값: 0.05)
    n_simulations : Monte Carlo 시뮬레이션 횟수 (기본값: 9999)
    """

    print(f"\n{'='*60}")
    print(f"[{col}] × [{target}] 검정")
    print(f"{'='*60}")

    # ── 1단계: 교차표 및 기대빈도 계산 ──────────────────────
    # [작업] pd.crosstab으로 교차표 생성
    #        chi2_contingency로 카이제곱 통계량, p-value, 자유도, 기대빈도 계산
    contingency_table = pd.crosstab(df[col], df[target])
    chi2, p, dof, expected = chi2_contingency(contingency_table)

    # ── 2단계: Cochran's rule 확인 ───────────────────────────
    # [작업] 조건 1: 기대빈도 1 미만인 셀이 없어야 함
    #       조건 2: 기대빈도 5 미만인 셀의 비율이 20% 이하여야 함
    #       -> 두 조건 중 하나라도 위반 시 Monte Carlo 시뮬레이션으로 바꿔서 수행
    total_cells = expected.size
    zero_cells = (expected < 1).sum()
    low_cells = (expected < 5).sum()
    low_cells_pct = low_cells / total_cells * 100

    print(f"\n[Cochran's rule 확인]")
    print(f"  전체 셀 수: {total_cells}")
    print(f"  기대빈도 1 미만 셀 수: {zero_cells}")
    print(f"  기대빈도 5 미만 셀 수: {low_cells} ({low_cells_pct:.1f}%)")

    # True: Monte Carlo 시뮬레이션 사용 / False: 카이제곱 검정 사용
    use_monte_carlo = (zero_cells > 0) or (low_cells_pct > 20)

    if use_monte_carlo:
        print(f"  ⚠️ Cochran's rule 위반 → Monte Carlo 시뮬레이션으로 대체")
    else:
        print(f"  ✅ Cochran's rule 충족 → 카이제곱 검정 진행")

    # ── 3단계: 카이제곱 검정 or Monte Carlo 시뮬레이션 ────────
    if use_monte_carlo:
        # [작업] scipy의 PermutationMethod를 사용하여 행/열 합계를 고정한 채
        #        n_simulations번 랜덤 교차표를 생성하여 p-value를 추정
        #        correction=False: Yates 연속성 보정은 Monte Carlo와 함께 사용 불가
        #        random_state=42: 재현 가능한 결과를 위해 시드 고정
        method = PermutationMethod(n_resamples=n_simulations, random_state=42)
        result = chi2_contingency(contingency_table, correction=False, method=method)
        chi2 = result.statistic
        p_val = result.pvalue

        print(f"\n[Monte Carlo 시뮬레이션 결과] (n={n_simulations:,})")
        print(f"  χ²     : {chi2:.4f}")
        print(f"  p-value: {p_val:.4f}")
        print(f"  자유도  : {dof}") # 자유도가 클수록 더 많은 범주 조합을 검정한다는 의미 
                                   # -> 카이제곱 통계량이 같아도 자유도에 따라 p-value가 달라진다고 함
    else:
        # [작업] p를 p_val로 명시적으로 할당하여 이후 코드에서 일관되게 사용
        p_val = p
        print(f"\n[카이제곱 검정 결과]")
        print(f"  χ²     : {chi2:.4f}")
        print(f"  p-value: {p_val:.4f}")
        print(f"  자유도  : {dof}")

    if p_val < alpha:
        print(f"  ✅ 유의미한 관계 있음 (p < {alpha})")
    else:
        print(f"  ❌ 유의미한 관계 없음 (p >= {alpha})")
        return None

    # ── 4단계: 크래머 V (효과 크기) ──────────────────────────
    # [작업] df* = min(r-1, c-1)에 따라 효과 크기 기준이 달라짐
    #        공식: √(χ² / (N · df*))
    #        df*별 효과 크기 기준은 Cohen(1988)의 w값(0.10, 0.30, 0.50)을
    #        V = w / √(df*) 공식에 대입하여 계산
    n = contingency_table.values.sum() # 전체 댓글 수
    df_star  = min(contingency_table.shape) - 1
    cramer_v = np.sqrt(chi2 / (n * df_star))

    small = 0.10 / np.sqrt(df_star)
    medium = 0.30 / np.sqrt(df_star)
    large = 0.50 / np.sqrt(df_star)

    print(f"\n[크래머 V (효과 크기)]")
    print(f"  df*    : {df_star}")
    print(f"  크래머 V: {cramer_v:.4f}")
    if cramer_v < small:
        print(f"  → 효과 매우 작음 ({small:.2f} 미만)")
    elif cramer_v < medium:
        print(f"  → 효과 작음 ({small:.2f} 이상 {medium:.2f} 미만)")
    elif cramer_v < large:
        print(f"  → 효과 중간 ({medium:.2f} 이상 {large:.2f} 미만)")
    else:
        print(f"  → 효과 큼 ({large:.2f} 이상)")

    # ── 5단계: 조정된 잔차 사후검정 ──────────────────────────
    # [작업] 조정된 잔차 = (관측빈도 - 기대빈도) / 표준오차
    #        표준오차 = sqrt(기대빈도 × (1 - 행비율) × (1 - 열비율))
    #        |조정된 잔차| > 1.96이면 유의미한 셀 (95% 신뢰수준)
    observed = contingency_table.values
    row_sums = observed.sum(axis=1, keepdims=True)
    col_sums = observed.sum(axis=0, keepdims=True)
    total = observed.sum()

    std_err = np.sqrt(expected * (1 - row_sums / total) * (1 - col_sums / total))
    adjusted_resid = (observed - expected) / std_err

    df_resid = pd.DataFrame(
        adjusted_resid,
        index=contingency_table.index,
        columns=contingency_table.columns
    ).round(2)

    print(f"\n[조정된 잔차 사후검정]")
    print(f"  기준: |조정된 잔차| > 1.96 → 유의미한 셀 (95% 신뢰수준)")
    display(df_resid)

    print(f"\n[유의미한 셀 요약]")
    for row in df_resid.index:
        for col_name in df_resid.columns:
            resid = df_resid.loc[row, col_name]
            if abs(resid) > 1.96:
                direction = "많음 (↑)" if resid > 0 else "적음 (↓)"
                print(f"  {row} × {col_name}: {resid} → 기대보다 {direction}")

    return {
        'chi2' : chi2,
        'p-value' : p_val,
        'df_star' : df_star,
        'cramer_v' : cramer_v,
        'use_monte_carlo': use_monte_carlo,
        'residuals' : df_resid
    }

In [ ]:
# ========================================
# [분석] 단일 변수별 감성 비율 검정 (범주형)
# ========================================

# [역할] 범주형 변수(cat_cols) 각각과 sentiment 간의 관계를 통계적으로 검정
# [근거] 카이제곱 or Monte Carlo 시뮬레이션 → 크래머 V → 조정된 잔차 순으로 수행
# [대상] video_format, production_quality, editing_pace,
#        color_mood, first_3sec, motion_graphic, background_style

cat_cols = [
    'video_format',         # 영상 유형
    'production_quality',   # 제작 퀄리티
    'editing_pace',         # 편집 속도
    'color_mood',           # 전반적인 영상의 색감
    'first_3sec',           # 영상이 시작되고 3초 이내의 요소
    'motion_graphic',       # 모션그래픽 사용 단계
    'background_style'      # 배경 스타일
]

cat_results = {} # 범주형 변수들의 통계검정 결과가 담길 딕셔너리

for col in cat_cols:
    result = chi2_test(df_it, col)
    cat_results[col] = result

In [ ]:
# 시각화 함수
def visualize_chi2(df, col, result, target='sentiment'):
    """
    Parameters
    ----------
    df     : 분석 데이터프레임
    col    : 검정한 범주형 변수 컬럼명
    result : chi2_test 함수의 반환값 (dict)
    target : 감성 컬럼명 (기본값: 'sentiment')
    """

    if result is None:
        print(f"[{col}] 유의미한 관계가 없어 시각화를 생략합니다.")
        return

    # ── 원본 보호: 함수 진입 시 무조건 copy ─────────────────
    # [근거] bool 타입 여부와 관계없이 함수 내부 수정이
    #        원본 DataFrame에 영향을 주지 않도록 방어적으로 복사
    df1 = df.copy()

    SENTIMENT_ORDER  = ["긍정", "중립", "부정"]
    SENTIMENT_COLORS = {"긍정": "#4CAF50", "중립": "#9E9E9E", "부정": "#F44336"}

    # ── bool 타입 컬럼 문자열 변환 ───────────────────────────
    # [작업] bool 타입 컬럼을 문자열로 변환하여
    #        y축 레이블이 숫자(0, 1)로 나타나는 문제 방지
    # [변경] copy()를 위로 올렸으므로 여기선 조건 분기만 남김
    if df1[col].dtype == bool:
        df1[col] = df1[col].map({True: 'True', False: 'False'})

    # ── 감성 비율 계산 ────────────────────────────────────────
    # [작업] 범주별 감성 비율 계산 후 긍정 비율 기준 오름차순 정렬
    grouped = (
        df1.groupby([col, target])
        .size()
        .reset_index(name='count')
    )
    total_per_group = grouped.groupby(col)['count'].transform('sum')
    grouped['ratio'] = grouped['count'] / total_per_group * 100

    pivot = (
        grouped
        .pivot_table(index=col, columns=target, values='ratio', fill_value=0)
        .reindex(columns=SENTIMENT_ORDER, fill_value=0)
    )

    # [작업] 긍정 비율 기준 오름차순 정렬
    pivot = pivot.sort_values('긍정', ascending=True)

    # ── 그래프 생성 ───────────────────────────────────────────
    n_rows = len(pivot)
    fig, axes = plt.subplots(1, 2, figsize=(16, max(5, n_rows * 0.5 + 2)))
    fig.suptitle(f"[{col}] × [{target}] 감성 분석", fontsize=15, fontweight='bold', y=1.01)

    # ── 차트 1: 누적 가로 막대차트 ───────────────────────────
    ax1  = axes[0]
    left = np.zeros(n_rows)

    for sentiment in SENTIMENT_ORDER:
        values = pivot[sentiment].values
        bars   = ax1.barh(
            pivot.index,
            values,
            left=left,
            color=SENTIMENT_COLORS[sentiment],
            label=sentiment,
            height=0.6,
        )
        for bar, val in zip(bars, values):
            if val == 0:
                continue
            x_center = bar.get_x() + bar.get_width() / 2
            y_center = bar.get_y() + bar.get_height() / 2

            if val >= 5:  # 내부 표시
                ax1.text(x_center, y_center, f"{val:.1f}%",
                         ha='center', va='center',
                         fontsize=8, color='white', fontweight='bold')
            else:          # 외부에 선 연결 (겹침 방지: 위/아래 교차 배치)
                offset_sign = 1 if SENTIMENT_ORDER.index(sentiment) % 2 == 0 else -1
                ax1.annotate(
                    f"{val:.1f}%",
                    xy=(x_center, y_center),
                    xytext=(x_center, y_center + offset_sign * 0.45),
                    arrowprops=dict(arrowstyle="-", color="gray", lw=1),
                    ha='center', va='center',
                    fontsize=8, color='black',
                )
        left += values

    ax1.set_xlim(0, 100)
    ax1.set_xlabel("비율 (%)", fontsize=11)

    # [작업] y축 레이블에 각 범주별 댓글 수 추가
    count_per_group = df1.groupby(col).size()
    new_labels = [f"{idx}\n(n={count_per_group[idx]:,})" for idx in pivot.index]
    ax1.set_yticklabels(new_labels, fontsize=9)

    ax1.set_title("감성 비율 (긍정 비율 기준 정렬)", fontsize=13)
    ax1.legend(
        loc='upper left',
        bbox_to_anchor=(0, 1.00),   # y축 위쪽으로 올리기
        ncol=3,                      # 범례를 가로로 나열
        fontsize=10,
    )
    ax1.xaxis.set_major_formatter(mticker.FormatStrFormatter("%d%%"))
    sns.despine(ax=ax1)

    # ── 차트 2: 조정된 잔차 히트맵 ──────────────────────────
    # [작업] RdYlGn 컬러맵 적용
    #        양수(초록): 해당 감성이 기대보다 자주 나타남
    #        음수(빨강): 해당 감성이 기대보다 적게 나타남
    #        |조정된 잔차| > 1.96인 셀만 유의미한 셀로 해석
    ax2      = axes[1]

    # [작업] bool 인덱스를 문자열로 변환 후 reindex하여 누적 막대차트와 순서 일치
    df1_resid = result['residuals'].copy()
    df1_resid.index = df1_resid.index.astype(str)
    df1_resid = df1_resid.reindex(pivot.index)

    sns.heatmap(
        df1_resid,
        ax=ax2,
        cmap='RdYlGn',
        center=0,
        annot=True,
        fmt='.2f',
        linewidths=0.5,
        cbar_kws={'label': '조정된 잔차'}
    )

    ax2.set_title("조정된 잔차 히트맵\n( |잔차| > 1.96, 유의미한 셀)\n(양수: 해당 감성이 기대보다 자주 나타남 / 음수: 해당 감성이 기대보다 적게 나타남)", fontsize=11)
    ax2.set_xlabel("감성", fontsize=11)
    ax2.set_ylabel("")

    plt.tight_layout()
    plt.show()

In [ ]:
for col in cat_cols:
    if cat_results[col] is not None:
        visualize_chi2(df_it, col, cat_results[col])

---
#### **B. 연속형 변수 → Kruskal-Wallis 검정**

In [ ]:
# ========================================
# [분석] 단일 변수별 감성 비율 검정 (연속형)
# ========================================

# [역할] 연속형 변수(person_ratio, face_ratio)와 sentiment 간의 관계를
#        Kruskal-Wallis 검정으로 통계적으로 검정
# [근거] 연속형 변수는 카이제곱 검정 적용 불가
#        → 정규성 가정이 필요 없는 비모수 검정인 Kruskal-Wallis 사용
# [대상] person_ratio, face_ratio

def kruskal_test(df, col, target='sentiment', alpha=0.05):
    """
    Parameters
    ----------
    df     : 분석 데이터프레임
    col    : 검정할 연속형 변수 컬럼명
    target : 감성 컬럼명 (기본값: 'sentiment')
    alpha  : 유의수준 (기본값: 0.05)
    """

    print(f"\n{'='*60}")
    print(f"[{col}] × [{target}] Kruskal-Wallis 검정")
    print(f"{'='*60}")

    # ── 1단계: 그룹별 데이터 분리 ────────────────────────────
    # [작업] sentiment 그룹별로 col 값을 분리하여 리스트로 저장
    groups = {}
    for sentiment in df[target].unique():
        groups[sentiment] = df[df[target] == sentiment][col].values

    print(f"\n[그룹별 기술통계]")
    for sentiment, values in groups.items():
        print(f"  {sentiment}: n={len(values)}, 중앙값={np.median(values):.4f}, 평균={np.mean(values):.4f}")

    # ── 2단계: Kruskal-Wallis 검정 ──────────────────────────
    # [작업] 세 감성 그룹(긍정/중립/부정) 간 col 분포에 차이가 있는지 검정
    # [귀무가설] 세 그룹 간 분포에 차이가 없다
    stat, p_val = kruskal(*groups.values())

    print(f"\n[Kruskal-Wallis 검정 결과]")
    print(f"  H 통계량: {stat:.4f}")
    p_str = f"{p_val:.4e}" if p_val < 0.0001 else f"{p_val:.4f}"
    print(f"  p-value : {p_str}")

    if p_val < alpha:
        print(f"  ✅ 유의미한 차이 있음 (p < {alpha})")
    else:
        print(f"  ❌ 유의미한 차이 없음 (p >= {alpha})")
        return None

    # ── 3단계: 효과크기 (eta-squared) ────────────────────────
    # [작업] Kruskal-Wallis 효과크기 계산
    # 공식: η² = (H - k + 1) / (n - k)
    # 해석 기준: 0.01 미만(매우 작음), 0.01~0.06(작음),
    #            0.06~0.14(중간), 0.14 이상(큼)
    n = len(df)
    k = len(groups)
    eta_squared = (stat - k + 1) / (n - k)

    print(f"\n[효과크기 (eta-squared)]")
    print(f"  η²: {eta_squared:.4f}")
    if eta_squared < 0.01:
        print(f"  → 효과 매우 작음 (0.01 미만)")
    elif eta_squared < 0.06:
        print(f"  → 효과 작음 (0.01 이상 0.06 미만)")
    elif eta_squared < 0.14:
        print(f"  → 효과 중간 (0.06 이상 0.14 미만)")
    else:
        print(f"  → 효과 큼 (0.14 이상)")

    # ── 4단계: Dunn's test 사후검정 ──────────────────────────
    # [작업] 어떤 그룹 간에 차이가 있는지 파악
    # [근거] Kruskal-Wallis는 전체적인 차이만 알려주므로
    #        사후검정으로 구체적인 그룹 쌍을 확인
    # [보정] Bonferroni: 다중비교 시 1종 오류 증가를 보정
    print(f"\n[Dunn's test 사후검정 (Bonferroni 보정)]")
    print(f"  ※ 테이블의 각 셀은 해당 그룹 쌍의 Bonferroni 보정된 p값")
    dunn_result = posthoc_dunn(
        df, val_col=col, group_col=target, p_adjust='bonferroni'
    )
    display(dunn_result.map(lambda x: f"{x:.4f}"))

    print(f"\n[유의미한 그룹 쌍 (보정된 p < {alpha})]")
    sentiments = list(dunn_result.columns)
    found = False
    for i in range(len(sentiments)):
        for j in range(i+1, len(sentiments)):
            p = dunn_result.loc[sentiments[i], sentiments[j]]
            if p < alpha:
                p_str = f"{p:.4e}" if p < 0.0001 else f"{p:.4f}"
                print(f"  {sentiments[i]} vs {sentiments[j]}: p={p_str} ✅")
                found = True
    if not found:
        print(f"  유의미한 그룹 쌍 없음")

    return {
        'stat':        stat,        # Kruskal-Wallis 검정통계량
        'p_value':     p_val,
        'eta_squared': eta_squared, # 효과크기
        'dunn_result': dunn_result  # Dunn's test 결과
    }

In [ ]:
# 함수 실행

cont_cols = [
    'person_ratio',  # 인물 등장 비율
    'face_ratio'     # 얼굴 등장 비율
]

cont_results = {}

for col in cont_cols:
    result = kruskal_test(df_it, col)
    cont_results[col] = result

In [ ]:
# ========================================
# [시각화] 연속형 변수별 감성 그룹 분포 비교
# ========================================

# [역할] person_ratio, face_ratio의 감성 그룹별 분포를
#        박스플롯과 바이올린 플롯으로 나란히 시각화
# [근거] 크루스칼-왈리스 검정 결과를 시각적으로 보완하여
#        그룹 간 분포 차이를 직관적으로 파악

SENTIMENT_ORDER  = ['긍정', '중립', '부정']
SENTIMENT_COLORS = {'긍정': '#4CAF50', '중립': '#9E9E9E', '부정': '#F44336'}

def visualize_kruskal(df, col, result, target='sentiment'):
    """
    Parameters
    ----------
    df     : 분석 데이터프레임
    col    : 검정한 연속형 변수 컬럼명
    result : kruskal_test 함수의 반환값 (dict)
    target : 감성 컬럼명 (기본값: 'sentiment')
    """

    if result is None:
        print(f"[{col}] 유의미한 관계가 없어 시각화를 생략합니다.")
        return

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(
        f"[{col}] × [{target}] 감성 그룹별 분포 비교\n"
        f"(η²={result['eta_squared']:.4f}, p={result['p_value']:.4e})",
        fontsize=13, fontweight='bold', y=1.02
    )

    palette = [SENTIMENT_COLORS[s] for s in SENTIMENT_ORDER]

    # ── 차트 1: 박스플롯 ─────────────────────────────────────
    # [작업] 감성 그룹별 col의 중앙값, IQR, 이상치 시각화
    ax1 = axes[0]
    sns.boxplot(
        data=df,
        x=target,
        y=col,
        order=SENTIMENT_ORDER,
        palette=palette,
        width=0.5,
        ax=ax1
    )
    ax1.set_title('박스플롯', fontsize=12)
    ax1.set_xlabel('감성', fontsize=10)
    ax1.set_ylabel(col, fontsize=10)
    ax1.set_ylim(-0.05, 1.05)
    sns.despine(ax=ax1)

    # ── 차트 2: 바이올린 플롯 ────────────────────────────────
    # [작업] 감성 그룹별 col의 분포 형태(KDE) + 박스플롯 겹쳐서 시각화
    # [근거] 데이터가 어디에 몰려있는지 박스플롯보다 더 세밀하게 파악 가능
    ax2 = axes[1]
    sns.violinplot(
        data=df,
        x=target,
        y=col,
        order=SENTIMENT_ORDER,
        palette=palette,
        inner='box',    # 바이올린 안에 박스플롯 겹치기
        ax=ax2
    )
    ax2.set_title('바이올린 플롯', fontsize=12)
    ax2.set_xlabel('감성', fontsize=10)
    ax2.set_ylabel(col, fontsize=10)
    ax2.set_ylim(-0.05, 1.05)
    sns.despine(ax=ax2)

    plt.tight_layout()
    plt.show()

In [ ]:
# 함수 실행 
for col in cont_cols:
    visualize_kruskal(df_it, col, cont_results[col])

---
### **두 가지 정보의 조합에서 댓글의 감성 분석하기**
- `video_format × production_quality` 분석
    - 영상 포맷에 따라 제작 퀄리티가 시청자의 긍정적인 반응에 미치는 영향이 달라지는지 확인하는 것이 목표임

- `video_format × editing_pace` 분석
    - 영상 포맷에 따라 편집 속도가 시청자의 긍정적인 반응에 미치는 영향이 달라지는지 확인하는 것이 목표임

- `video_format × first_3sec` 분석
    - 영상 포맷에 따라 효과적인 오프닝 구성이 달라지는지, 어떤 조합이 시청자의 긍정적인 반응을 이끌어내기에 유리한지 확인하는 것이 목표임 

- `color_mood × background_style` 분석
    - 색감과 배경 스타일의 조합이 시청자의 긍정적인 반응에 미치는 영향을 확인하는 것이 목표임

- `editing_pace × color_mood` 분석
    - 편집 속도와 색감의 조합이 시청자의 긍정적인 반응에 미치는 영향을 확인하는 것이 목표임

In [ ]:
# ================================================
# [분석] 두 범주형 변수 교차 분석 함수 (범용)
# ================================================

# [역할] 두 범주형 변수의 조합이 댓글 긍정 비율에 미치는 영향을 분석
# [근거] 두 범주형 변수 간 전체 연관성을 카이제곱으로 검정하고,
#        조합별 긍정률을 히트맵으로 시각화하여
#        긍정 반응을 이끌어내기에 유리한 조합을 도출

def analyze_cross(df_input, col1, col2, label, target='sentiment'):
    """
    Parameters
    ----------
    df_input : 분석 대상 데이터프레임
    col1     : 행 변수 컬럼명 (히트맵 행)
    col2     : 열 변수 컬럼명 (히트맵 열)
    label    : 출력 및 제목에 사용할 레이블 (예: 'IT 숏폼')
    target   : 감성 컬럼명 (기본값: 'sentiment')
    """

    print(f"\n{'='*60}")
    print(f"[{label}] {col1} × {col2} 교차 분석")
    print(f"{'='*60}")

    # ── 0. 분석용 컬럼 추출 ─────────────────────────────────────
    # [작업] 분석에 필요한 컬럼만 추출
    df_cross = df_input[[col1, col2, target]].copy()

    print(f"\n[{col1} 분포]")
    print(df_cross[col1].value_counts())
    print(f"\n[{col2} 분포]")
    print(df_cross[col2].value_counts())

    # ── 1. 긍정 여부 컬럼 생성 ──────────────────────────────────
    # [작업] sentiment == '긍정'이면 1, 아니면 0
    df_cross['is_positive'] = (df_cross[target] == '긍정').astype(int)

    # ── 2. (col1 × col2)별 긍정 비율 집계 ───────────────────────
    agg = (
        df_cross.groupby([col1, col2])['is_positive']
        .agg(positive_count='sum', total='count')
        .reset_index()
    )
    agg['positive_ratio'] = agg['positive_count'] / agg['total']

    print(f"\n=== ({col1} × {col2})별 긍정 비율 ===")
    display(agg.sort_values([col1, col2]).reset_index(drop=True))

    # ── 3. 통계 검정: 전체 교차표 카이제곱 ───────────────────────
    # [작업] col1 × col2 전체 교차표로 카이제곱 검정
    # [근거] 두 변수 모두 다중 범주형이라 전체 연관성을 한 번에 검정하는 방식을 사용
    # [주의] 댓글 수(total) 기준으로 교차표 생성
    #        → 각 셀: 해당 조합의 전체 댓글 수
    ct = pd.crosstab(df_cross[col1], df_cross[col2])

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        chi2, p, dof, expected = chi2_contingency(ct)

    # 효과크기: Cramér's V 계산
    # [근거] 카이제곱은 표 크기에 따라 값이 달라지므로
    #        표 크기를 보정한 Cramér's V를 효과크기로 사용
    # 계산식: V = sqrt(χ² / (n × (min(행수, 열수) - 1)))
    # df*에 따라 효과 크기 기준이 달라짐 (Cohen 1988)
    n_total   = df_cross.shape[0]
    min_dim   = min(ct.shape) - 1
    cramers_v = round(np.sqrt(chi2 / (n_total * min_dim)), 4)

    small  = 0.10 / np.sqrt(min_dim)
    medium = 0.30 / np.sqrt(min_dim)
    large  = 0.50 / np.sqrt(min_dim)

    if cramers_v < small:
        v_interpret = f"효과 매우 작음 ({small:.2f} 미만)"
    elif cramers_v < medium:
        v_interpret = f"효과 작음 ({small:.2f} 이상 {medium:.2f} 미만)"
    elif cramers_v < large:
        v_interpret = f"효과 중간 ({medium:.2f} 이상 {large:.2f} 미만)"
    else:
        v_interpret = f"효과 큼 ({large:.2f} 이상)"

    print(f"\n=== 통계 검정 결과 ===")
    print(f"카이제곱 통계량 : {chi2:.4f}")
    p_str = f"{p:.4e}" if p < 0.0001 else f"{p:.4f}"
    print(f"p-value         : {p_str}")
    print(f"자유도          : {dof}")
    print(f"Cramér's V      : {cramers_v} → {v_interpret}")
    print(f"통계적 유의성   : {'유의함 (p < 0.05)' if p < 0.05 else '유의하지 않음 (p >= 0.05)'}")

    # ── 4. 피벗: 히트맵용 긍정률 테이블 생성 ─────────────────────
    # [작업] 행: col1, 열: col2, 값: positive_ratio
    # [주의] 댓글 수가 30개 미만인 셀은 NaN으로 처리하여 히트맵에서 제외
    #        → 샘플이 너무 작은 조합은 신뢰하기 어려움

    # 셀별 댓글 수 피벗 (30개 미만 필터링용)
    total_pivot = agg.pivot_table(
        index=col1,
        columns=col2,
        values='total'
    )

    # 긍정률 피벗
    ratio_pivot = agg.pivot_table(
        index=col1,
        columns=col2,
        values='positive_ratio'
    )

    # 댓글 수 30개 미만인 셀은 NaN으로 마스킹
    # [근거] 샘플이 너무 작으면 긍정률이 극단적으로 나올 수 있음
    ratio_pivot_masked = ratio_pivot.where(total_pivot >= 30)

    print(f"\n=== 조합별 긍정률 테이블 (댓글 수 30개 미만 셀은 NaN) ===")
    display(ratio_pivot_masked.round(4))

    # ── 5. 히트맵 시각화 ─────────────────────────────────────────
    # [작업] 조합별 긍정률을 히트맵으로 시각화
    # [근거] 수치 비교표보다 색상으로 패턴을 직관적으로 파악할 수 있음
    #        NaN 셀은 회색으로 표시 (데이터 부족)

    fig, ax = plt.subplots(figsize=(14, 7))

    # NaN 셀 위치 마스크 생성
    nan_mask = ratio_pivot_masked.isna()

    # 배경을 먼저 회색으로 깔기 (NaN 셀이 회색으로 보이도록)
    ax.set_facecolor('#D3D3D3')

    sns.heatmap(
        ratio_pivot_masked,
        annot=True,
        fmt='.2%',
        cmap='RdYlGn',
        vmin=0.5,
        vmax=1.0,
        linewidths=0.5,
        linecolor='gray',
        mask=nan_mask,
        ax=ax,
        cbar_kws={'label': '긍정 댓글 비율'}
    )

    ax.set_title(
        f'[{label}] {col1} × {col2} 긍정 댓글 비율\n'
        f'(Cramér\'s V={cramers_v} ({v_interpret}), p={p_str} / 회색=데이터 부족(30개 미만))',
        fontsize=12
    )
    ax.set_xlabel(col2, fontsize=10)
    ax.set_ylabel(col1, fontsize=10)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right', fontsize=9)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=9)

    plt.tight_layout()
    plt.show()

    return ratio_pivot_masked

In [ ]:
# 함수 실행
cross_pairs = [
    ('video_format', 'production_quality'),
    ('video_format', 'editing_pace'),
    ('video_format', 'first_3sec'),
    ('color_mood',   'background_style'),
    ('editing_pace', 'color_mood'),
]

cross_results_it = {}

for col1, col2 in cross_pairs:
    result = analyze_cross(df_it, col1, col2, label='IT 숏폼')
    cross_results_it[(col1, col2)] = result